In [0]:
from pyspark.sql.functions import col, sum, when, count, lit, expr
from pyspark.sql.window import Window
from pyspark.sql import Row

In [0]:
data = spark.table("workspace.default.steam_game_output")
df = data
df = df.toDF(*[c.replace(".", "_") for c in df.columns])

#### Most missing values in this dataset are not missing at all — they represent absence of a feature, such as platform support, genre applicability, release metadata. Imputation would distort the analysis.

# Platform analysis

## Are most games available on Windows/Mac/Linux instead?

In [0]:
platform_counts = df.agg(
    sum(when(col("platforms_windows") == True, 1).otherwise(0)).alias("Windows"),
    sum(when(col("platforms_mac") == True, 1).otherwise(0)).alias("Mac"),
    sum(when(col("platforms_linux") == True, 1).otherwise(0)).alias("Linux")
)


In [0]:
platform_counts.printSchema()

root
 |-- Windows: long (nullable = true)
 |-- Mac: long (nullable = true)
 |-- Linux: long (nullable = true)



In [0]:
df_ratios = platform_counts.select(
    (col("windows") / (col("mac") + col("linux"))).alias("win_vs_others"),
    (col("mac") / col("linux")).alias("mac_vs_linux"),
    (col("windows") / col("linux")).alias("win_vs_linux")
)
display(df_ratios)

win_vs_others,mac_vs_linux,win_vs_linux
2.6227623892971548,1.5098131946086546,6.5826436509813195


In [0]:
windows = df_ratios.collect()[0]["win_vs_others"]
mac = df_ratios.collect()[0]["mac_vs_linux"]
linux = df_ratios.collect()[0]["win_vs_linux"]

In [0]:
df_platform_summary = spark.createDataFrame([
    Row(platform="Windows", count=windows),
    Row(platform="Mac", count=mac),
    Row(platform="Linux", count=linux)
])
display(df_platform_summary)

platform,count
Windows,2.6227623892971548
Mac,1.5098131946086546
Linux,6.5826436509813195


In [0]:
df_long = df.select(
    "genre",
    expr("stack(3, 'Windows', platforms_windows, 'Mac', platforms_mac, 'Linux', platforms_linux) as (platform, available)")
)

In [0]:
df_long = df_long.filter(col("available") == True)

In [0]:
display(df_long)

genre,platform,available
Action,Windows,true
"Action, Adventure, Indie",Windows,true
"Adventure, Indie, RPG, Strategy",Windows,true
"Action, Indie, Simulation, Strategy",Windows,true
"Action, Casual, Indie, Simulation",Windows,true
"Action, Adventure, Indie, RPG",Windows,true
"Adventure, Indie, RPG, Strategy",Windows,true
"Action, Adventure, Casual, Free to Play, Massively Multiplayer",Windows,true
"Casual, Indie",Windows,true
"Indie, RPG",Windows,true


Databricks visualization. Run in Databricks to view.

#### Game availability were found to be more than twice the sum of both Mac and Linux, while Mac had ~50% more games released than Linux on this dataset

## Do certain genres tend to be preferentially available on certain platforms?

In [0]:
df_platform = df.select(
    "genre",
    col("platforms_windows").alias("windows"),
    col("platforms_mac").alias("mac"),
    col("platforms_linux").alias("linux")
)

In [0]:
windows_df = df_platform.select(
    "genre",
    lit("windows").alias("platform"),
    col("windows").alias("available")
)

mac_df = df_platform.select(
    "genre",
    lit("mac").alias("platform"),
    col("mac").alias("available")
)

linux_df = df_platform.select(
    "genre",
    lit("linux").alias("platform"),
    col("linux").alias("available")
)

df_long = windows_df.union(mac_df).union(linux_df)

In [0]:
df_long = df_long.filter(col("available") == True)

In [0]:
counts = (
    df_long.groupBy("genre", "platform")
    .agg(count("*").alias("n_games"))
)

In [0]:
window = Window.partitionBy("genre")

counts = counts.withColumn(
    "total_genre",
    sum("n_games").over(window)
)

counts = counts.withColumn(
    "platform_share",
    col("n_games") / col("total_genre")
)

display(counts)

genre,platform,n_games,total_genre,platform_share
,windows,159,256,0.62109375
,mac,55,256,0.21484375
,linux,42,256,0.1640625
Accounting,windows,4,6,0.6666666666666666
Accounting,mac,2,6,0.3333333333333333
"Accounting, Animation & Modeling, Audio Production, Design & Illustration, Education, Photo Editing, Software Training, Utilities, Video Production, Web Publishing",windows,2,2,1.0
"Accounting, Animation & Modeling, Audio Production, Design & Illustration, Education, Photo Editing, Software Training, Utilities, Video Production, Web Publishing, Game Development",windows,2,3,0.6666666666666666
"Accounting, Animation & Modeling, Audio Production, Design & Illustration, Education, Photo Editing, Software Training, Utilities, Video Production, Web Publishing, Game Development",mac,1,3,0.3333333333333333
"Accounting, Education, Software Training, Utilities, Early Access",windows,1,1,1.0
"Accounting, Utilities",windows,1,1,1.0


In [0]:
heat.printSchema()

root
 |-- genre: string (nullable = true)
 |-- platform: string (nullable = false)
 |-- n_games: long (nullable = false)
 |-- total_genre: long (nullable = true)
 |-- platform_share: double (nullable = true)
 |-- total_genre: long (nullable = true)



In [0]:
top_genres = (
    counts.groupBy("genre")
    .agg(sum("n_games").alias("total_genre"))
    .orderBy("total_genre", ascending=False)
    .limit(12)
)
heat = counts.join(top_genres, "genre")
display(heat)

genre,platform,n_games,total_genre,platform_share,total_genre
Action,windows,1632,1996,0.8176352705410822,1996
Action,mac,209,1996,0.10470941883767534,1996
Action,linux,155,1996,0.07765531062124248,1996
"Action, Adventure, Indie",windows,2783,3872,0.71875,3872
"Action, Adventure, Indie",mac,612,3872,0.15805785123966942,3872
"Action, Adventure, Indie",linux,477,3872,0.12319214876033058,3872
"Action, Casual, Indie",windows,1914,2717,0.7044534412955465,2717
"Action, Casual, Indie",mac,452,2717,0.16635995583364005,2717
"Action, Casual, Indie",linux,351,2717,0.1291866028708134,2717
"Action, Indie",windows,3460,4892,0.7072771872444807,4892


Databricks visualization. Run in Databricks to view.

#### Windows has a higher availability of Action and Casual games compared to Mac and Linux. 
#### Reflected through all the other genres, there is an overall dominance of Windows in the dataset.

In [0]:
df_platform.write.mode("overwrite").saveAsTable("platform_summary")

In [0]:
heat_clean = heat.drop(heat.columns[5])
heat_clean.write.mode("overwrite").saveAsTable("platform_heat")

In [0]:
df_long.write.mode("overwrite").saveAsTable("platform_long")